# 🦙 Ollama Server + ngrok on Colab GPU

This notebook:
1. Installs Ollama on Colab
2. Pulls `qwen2.5-coder:7b` (runs on ~8 GB VRAM — fits Colab T4)
3. Starts the Ollama server
4. Exposes port **11434** publicly via **ngrok**
5. Prints the URL → set it in your local PowerShell before running `agent.py`

---
**Before you start:**  
Get a free ngrok token from 👉 https://dashboard.ngrok.com/get-started/your-authtoken  
Paste it in **Cell 2** below.

In [1]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 108 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (489 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
# ── Cell 1: Install Ollama & pyngrok ─────────────────────────
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q pyngrok
print("✅ Ollama + pyngrok installed")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
✅ Ollama + pyngrok installed


In [3]:
!ollama --version

In [4]:
# ── Cell 2: Set ngrok auth token ─────────────────────────────
# 👉 Get yours free from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "3B5iZU9vm5V6ZPmhba5LDhRg6ed_7fxZLjadWxECmEhacN2LK"   # <── paste token here

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTH_TOKEN
print("✅ ngrok token set")

✅ ngrok token set


In [5]:
# ── Cell 3: Start Ollama server in background ─────────────────
import subprocess, time, requests, os

MODEL = "qwen2.5-coder:7b"
PORT  = 11434

# Allow connections from outside localhost (needed for ngrok)
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0"

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for Ollama to be ready
print("⏳ Waiting for Ollama server", end="")
for _ in range(30):
    time.sleep(1)
    try:
        r = requests.get(f"http://localhost:{PORT}", timeout=2)
        if r.status_code == 200:
            print("\n✅ Ollama server is UP!")
            break
    except Exception:
        print(".", end="", flush=True)
else:
    print("\n❌ Ollama did not start — check logs:")
    out, _ = ollama_proc.communicate(timeout=5)
    print(out.decode()[-2000:])

⏳ Waiting for Ollama server
✅ Ollama server is UP!


In [6]:
# ── Cell 4: Pull the model ────────────────────────────────────
# qwen2.5-coder:7b ≈ 4.7 GB download — takes ~3 min on Colab
print(f"📥 Pulling {MODEL} ...")
result = subprocess.run(["ollama", "pull", MODEL], capture_output=True, text=True)
print(result.stdout[-1000:] if result.stdout else result.stderr[-1000:])
print(f"✅ Model ready: {MODEL}")

📥 Pulling qwen2.5-coder:7b ...
ng 60e05f210007: 100% ▕██████████████████▏ 4.7 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling 1e65450c3067: 100% ▕██████████████████▏ 1.6 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling d9bb33f27869: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest ⠇ pulling manifest 
pulling 60e05f210007: 100% ▕██████████████████▏ 4.7 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling 1e65450c3067: 100% ▕██████████████████▏ 1.6 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling d9bb33f27869: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 

✅ Model ready: qwen2.5-coder:7b


In [7]:
# ── Cell 5: Expose via ngrok & print URL ─────────────────────
from pyngrok import ngrok

tunnel     = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

print()
print("╔══════════════════════════════════════════════════════════════╗")
print("║   🦙 YOUR OLLAMA SERVER IS LIVE                              ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║   URL : {public_url:<54}║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║   Run this in your LOCAL PowerShell before agent.py:         ║")
print(f"║   $env:OLLAMA_NGROK_URL = \"{public_url}\"")
print("║   python agent.py                                            ║")
print("╚══════════════════════════════════════════════════════════════╝")

                                                                                                    
╔══════════════════════════════════════════════════════════════╗
║   🦙 YOUR OLLAMA SERVER IS LIVE                              ║
╠══════════════════════════════════════════════════════════════╣
║   URL : https://autotomic-buckishly-adele.ngrok-free.dev      ║
╠══════════════════════════════════════════════════════════════╣
║   Run this in your LOCAL PowerShell before agent.py:         ║
║   $env:OLLAMA_NGROK_URL = "https://autotomic-buckishly-adele.ngrok-free.dev"
║   python agent.py                                            ║
╚══════════════════════════════════════════════════════════════╝


In [8]:
# ── Cell 6: Quick test ───────────────────────────────────────
import requests, json

resp = requests.post(
    public_url + "/api/chat",
    json={
        "model":    MODEL,
        "messages": [{"role": "user", "content": "Write a Python hello world"}],
        "stream":   False,
    }
)
data = resp.json()
print("✅ Test response:")
print(data["message"]["content"])

✅ Test response:
Certainly! Here's a simple "Hello, World!" program in Python:

```python
print("Hello, World!")
```

To run this program, you can save it to a file with a `.py` extension, for example, `hello.py`, and then execute it using a Python interpreter. For instance, if you have Python installed on your system, you can run the following command in your terminal:

```sh
python hello.py
```

This will output:

```
Hello, World!
```


In [9]:
# ── Cell 7: Keep-alive (prevents Colab idle timeout) ─────────
# Leave this running while you use agent.py locally.
# Interrupt kernel (■) to stop.
import time
print(f"🔁 Keeping Ollama alive at {public_url}")
print("   Interrupt kernel (■) to stop.")
try:
    while True:
        time.sleep(60)
        print("💓 alive...")
except KeyboardInterrupt:
    ollama_proc.terminate()
    ngrok.disconnect(public_url)
    print("🛑 Server stopped.")

🔁 Keeping Ollama alive at https://autotomic-buckishly-adele.ngrok-free.dev
   Interrupt kernel (■) to stop.
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...
💓 alive...


: 